<a href="https://colab.research.google.com/github/LCaravaggio/NLP/blob/main/notebooks/09_Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Vamos a usar [Ollama](https://ollama.com/) con un modelo de lenguaje local.

* Es un programa que permite descargar y ejecutar modelos de lenguaje de manera local.
* Desde la terminal, `ollama pull llama3.1` descarga un modelo y `ollama run llama3.1` abre un chat interactivo.
* También expone una API local en `localhost:11434` que se puede consumir desde cualquier lenguaje. En Python, el paquete `ollama` provee una interfaz directa con funciones como `ollama.chat()` y `ollama.generate()`.

Ollama se instala y ejecuta directamente en la instancia de Colab.
Recomendamos usar un runtime con GPU.

-----------------------

Tarea: responder donde dice **PREGUNTA**

## Configuración del entorno

In [1]:
!sudo apt-get install -y zstd pciutils # dependencias
!curl -fsSL https://ollama.com/install.sh | sh # Ollama

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils zstd
0 upgraded, 4 newly installed, 0 to remove and 33 not upgraded.
Need to get 1,025 kB of archives.
After this operation, 3,604 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 pci.ids all 0.0~2024.03.31-1ubuntu0.1 [275 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libpci3 amd64 1:3.10.0-2build1 [36.5 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 pciutils amd64 1:3.10.0-2build1 [69.7 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 1,025 kB in 1s (760 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be 

In [2]:
!pip install -qU ollama datasets watermark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 99.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [3]:
%reload_ext watermark

In [4]:
%watermark -vmp ollama,datasets,numpy,pandas

Python implementation: CPython
Python version       : 3.13.15
IPython version      : 7.34.0

ollama  : 0.6.3
datasets: 5.1.0
numpy   : 2.1.3
pandas  : 2.2.3

Compiler    : GCC 13.3.0
OS          : Linux
Release     : 6.6.122+
Machine     : x86_64
Processor   : x86_64
CPU cores   : 2
Architecture: 64bit



Hay que iniciar el servidor Ollama en segundo plano y dejarlo corriendo

In [5]:
import subprocess
import time

# matar cualquier proceso anterior y reiniciar el servidor
!pkill -f "ollama serve" 2>/dev/null

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

print(f"Ollama server iniciado (PID: {process.pid})")

^C
Ollama server iniciado (PID: 2823)


In [6]:
# Modelo acorde a los recursos de Colab
MODEL_NAME = "gemma3:1b"

!ollama pull {MODEL_NAME}

In [7]:
import ollama

models = ollama.list()
for m in models['models']:
    print(f"{m['model']}  ({m['size'] / 1e9:.1f} GB)")

gemma3:1b  (0.8 GB)


En [ollama.com/library](https://ollama.com/library) están todos los modelos con sus variantes.

Para comparar rendimiento entre modelos:

* [Chatbot Arena Leaderboard](https://lmarena.ai/) ranquea modelos según preferencia humana
* [Artificial Analysis](https://artificialanalysis.ai/) publica índices de inteligencia que agregan benchmarks independientes, incluyendo modelos chicos y abiertos

## Introducción al uso de Ollama

Usamos un LLM local para generar texto a partir de prompts.

In [8]:
%%time
response = ollama.chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "¿Qué es una chocotorta?"}]
)

CPU times: user 5.92 ms, sys: 1.5 ms, total: 7.42 ms
Wall time: 2min 1s


In [9]:
print(response)

model='gemma3:1b' created_at='2026-10-16T14:50:23.371583127Z' done=True done_reason='stop' total_duration=121376718056 load_duration=52963912167 prompt_eval_count=17 prompt_eval_duration=66120285000 eval_count=307 eval_duration=2285997000 message=Message(role='assistant', content='La palabra "chocotorta" es una palabra inventada de la serie de libros "El Principito" de Antoine de Saint-Exupéry. **No es un término real o conocido en la lingüística o en la cultura popular.**\n\nLa palabra fue creada por el autor como una *pequeña crítica* sobre la superficialidad y la falta de profundidad de la vida adulta.  En la historia, el Principito la utiliza para referirse a alguien que se preocupa demasiado por lo que otros piensan de él y se centra en su propio estatus y apariencia.\n\n**En resumen, es una palabra inventada que se usa para describir a alguien que es preocupado por las apariencias y el juicio de los demás.**\n\n**Aquí hay algunos puntos clave sobre su origen:**\n\n* **Origen:**  

In [10]:
print(response['message']['content'])

La palabra "chocotorta" es una palabra inventada de la serie de libros "El Principito" de Antoine de Saint-Exupéry. **No es un término real o conocido en la lingüística o en la cultura popular.**

La palabra fue creada por el autor como una *pequeña crítica* sobre la superficialidad y la falta de profundidad de la vida adulta.  En la historia, el Principito la utiliza para referirse a alguien que se preocupa demasiado por lo que otros piensan de él y se centra en su propio estatus y apariencia.

**En resumen, es una palabra inventada que se usa para describir a alguien que es preocupado por las apariencias y el juicio de los demás.**

**Aquí hay algunos puntos clave sobre su origen:**

* **Origen:**  Fue acuñada por Antoine de Saint-Exupéry en una escena en "El Principito" para representar la excesiva preocupación por la apariencia.
* **Uso en la historia:**  En la historia, el Principito la usa para referirse a un hombre que se preocupa demasiado por las tendencias de moda.
* **No es 

**PREGUNTA 1**: ¿Qué caracteriza a la respuesta anterior? ¿Por qué?

* Ollama carga el modelo en memoria (RAM o VRAM) de manera automática con el primer request.
* El modelo queda cargado durante 5 minutos por defecto; si no recibe requests en ese tiempo, se libera de la memoria automáticamente.
* Para ver qué modelos están cargados y si corren en GPU o CPU (columna `PROCESSOR`), se puede usar `ollama ps`.
* Para liberar un modelo de la memoria manualmente, `ollama stop <modelo>`.
* Si se hace un request con un modelo distinto, Ollama lo carga; si no entra en memoria junto con el anterior, primero libera el anterior.

A veces es útil usar "system prompts" para alterar el comportamiento del modelo.

In [11]:
system_instruction = """\
Sos un experto riguroso, breve y conciso. Respondés siempre con lo justo y necesario, sin listas, títulos ni negritas.
"""
print(system_instruction)

Sos un experto riguroso, breve y conciso. Respondés siempre con lo justo y necesario, sin listas, títulos ni negritas.



In [12]:
%%time
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": "¿Qué es una chocotorta?"}
    ]
)

CPU times: user 2.19 ms, sys: 60 µs, total: 2.25 ms
Wall time: 244 ms


In [13]:
print(response['message']['content'])

Es un tipo de dulce de almendras, a menudo con un núcleo de nuez y un relleno de almendras o avena.


In [14]:
# función para hacer un print en formato markdown.
import textwrap

from IPython.display import Markdown

def to_markdown(text):
    text = text.replace('•', '  *').replace("\n", "\n\n")
    # predicate: hack para indentar todas las líneas
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [15]:
response = ollama.chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "¿Por qué Sibarita es tan rica?"}]
)
to_markdown(response['message']['content'])

> Esta es una pregunta que genera mucha curiosidad y un poco de debate. La razón por la que Sibarita es tan rica es un mito que se popularizó en internet a finales de los 90 y principios de los 2000. La historia es lo siguiente:
> 
> 
> 
> **La Historia:**
> 
> 
> 
> * **Un Viaje de Exploración:** En la década de 1990, un joven turista, un hombre llamado Sibarita, se fue a viajar a América Latina en busca de un "tesoro oculto".
> 
> * **El Tesoro: Una Cueva en Guatemala:** El tesoro se decía que estaba escondido en una cueva en Guatemala, y la leyenda cuenta que el tesoro era de un famoso pirata llamado el "Padre".
> 
> * **La Muerte de Sibarita y el Tesoro:** Sibarita murió en Guatemala.  La historia dice que su cuerpo fue encontrado en una cueva y la única pista que encontró era un pequeño cofre con un pergamino que decía: "La riqueza de la cueva es la luz de las estrellas".
> 
> * **La Consecuencia del Pergamino:**  Al intentar leer el pergamino, se cree que el cofre se abrió, revelando un complejo sistema de espejos y ventanas que revelaban una serie de plátanos y, finalmente, el tesoro.
> 
> 
> 
> **¿Por qué es un mito?**
> 
> 
> 
> * **Origen Falso:** La historia de Sibarita y el tesoro es completamente falsa.  No hay evidencia histórica que respalde la existencia de un pirata llamado el "Padre" ni de un tesoro escondido en una cueva en Guatemala.
> 
> * **Un Juego de la Creatividad:** El mito se popularizó como una forma de **juego de la creatividad y el ingenio**. La historia, con su elementos sobrenaturales, se vuelve atractiva y memorable.
> 
> 
> 
> **En resumen, la historia de Sibarita es un mito que ha sido popularizada por internet, pero no tiene ninguna base en la realidad.**
> 
> 
> 
> **En conclusión, Sibarita es rica por una historia inventada que se ha convertido en un fenómeno viral.**
> 
> 
> 
> ¿Te interesa saber algo más sobre este mito?  Podríamos hablar sobre:
> 
> 
> 
> *   ¿Por qué la historia de Sibarita se popularizó?
> 
> *   ¿Cómo ha sido utilizada en la cultura popular?
> 
> *   ¿Qué tipo de imágenes y elementos de la historia se utilizan para transmitir la idea de riqueza?

In [16]:
# Ollama devuelve métricas en cada respuesta.

ns_to_s = 1e-9 # duraciones en nanosegundos

prompt_tokens = response['prompt_eval_count']
generated_tokens = response['eval_count']
prompt_time = response['prompt_eval_duration'] * ns_to_s
generation_time = response['eval_duration'] * ns_to_s
total_time = response['total_duration'] * ns_to_s

print(f"Tokens del prompt:    {prompt_tokens}")
print(f"Tokens generados:     {generated_tokens}")
print(f"Tiempo de prompt:     {prompt_time:.2f}s ({prompt_tokens / prompt_time:.1f} tokens/s)")
print(f"Tiempo de generación: {generation_time:.2f}s ({generated_tokens / generation_time:.1f} tokens/s)")
print(f"Tiempo total:         {total_time:.2f}s")

Tokens del prompt:    18
Tokens generados:     510
Tiempo de prompt:     0.02s (930.8 tokens/s)
Tiempo de generación: 3.65s (139.6 tokens/s)
Tiempo total:         3.68s


**PREGUNTA 2** : ¿Qué diferencia hay entre "tiempo de prompt" y "tiempo de generación"? ¿A que se debe la diferencia?

Con el parámetro `options` podemos ajustar parámetros de generación, como `temperature` o `num_predict` (número máximo de tokens en el output).

In [17]:
response = ollama.chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Escribir una canción de cancha para la selección de Uruguay."}],
    options={
        "num_predict": 120,
        "temperature": 1.0,
        "seed": 33, # resultados determinísticos
    }
)

In [18]:
to_markdown(response['message']['content'])

> ¡Absolutamente! Aquí tienes una canción de cancha para Uruguay, inspirada en la historia y la energía de la selección, con un toque de nostalgia y coraje:
> 
> 
> 
> **Título: La Sangre del Río**
> 
> 
> 
> (Intro - ritmo enérgico, batería con un toque de guitarra acústica y algunos toques de saxo)
> 
> 
> 
> **(Verso 1)**
> 
> El viento sopla, la tierra tiembla,
> 
> un rugido ancestral que nos llama.
> 
> De la península, de la laguna,
> 
> nuestro espíritu es una llama.
> 
> Desde el barro, una

**PREGUNTA 3**: ¿Cómo podemos obtener resultados determinísticos si temperature=1.0?

In [19]:
# Con ollama show <modelo> vemos las características de un modelo descargado.
!ollama show {MODEL_NAME}

  Model
    architecture        gemma3     
    parameters          999.89M    
    context length      32768      
    embedding length    1152       
    quantization        Q4_K_M     
    requires            0.30.0     

  Capabilities
    completion    

  Parameters
    top_p          0.95               
    stop           "<end_of_turn>"    
    temperature    1                  
    top_k          64                 

  License
    Gemma Terms of Use                  
    Last modified: February 21, 2024    
    ...                                 



In [20]:
# También desde python
info = ollama.show(MODEL_NAME)

for key, value in info.modelinfo.items():
    print(f"{key}: {value}")

gemma3.attention.head_count: 4
gemma3.attention.head_count_kv: 1
gemma3.attention.key_length: 256
gemma3.attention.layer_norm_rms_epsilon: 1e-06
gemma3.attention.sliding_window: 512
gemma3.attention.value_length: 256
gemma3.block_count: 26
gemma3.context_length: 32768
gemma3.embedding_length: 1152
gemma3.feed_forward_length: 6912
gemma3.rope.freq_base: 1000000
gemma3.rope.freq_base_swa: 10000
general.architecture: gemma3
general.base_model.0.name: Gemma 3 1b Pt
general.base_model.0.organization: Google
general.base_model.0.repo_url: https://huggingface.co/google/gemma-3-1b-pt
general.base_model.count: 1
general.basename: google-gemma-3
general.file_type: 15
general.finetune: it
general.license: gemma
general.parameter_count: 999885952
general.quantization_version: 2
general.sampling.top_k: 64
general.sampling.top_p: 0.95
general.size_label: 1B
general.tags: ['text-generation']
general.type: model
tokenizer.ggml.add_bos_token: True
tokenizer.ggml.add_eos_token: False
tokenizer.ggml.add_

## Clasificación zero-shot

Usemos el LLM para clasificar reviews.

In [21]:
from datasets import load_dataset

dataset = load_dataset('cornell-movie-review-data/rotten_tomatoes')

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 92.2kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [22]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


In [23]:
id2label = {0: 'negative', 1: 'positive'}
label2id = {'negative': 0, 'positive': 1}

In [24]:
examples = dataset['test'].shuffle(seed=36).select(range(5))
examples[:]

{'text': ["it's like an old warner bros . costumer jived with sex -- this could be the movie errol flynn always wanted to make , though bette davis , cast as joan , would have killed him .",
  "[lin chung's] voice is rather unexceptional , even irritating ( at least to this western ear ) , making it awfully hard to buy the impetus for the complicated love triangle that develops between the three central characters .",
  "as his circle of friends keeps getting smaller one of the characters in long time dead says 'i'm telling you , this is f * * * ed' . maybe he was reading the minds of the audience .",
  "what emerges is an unsettling picture of childhood innocence combined with indoctrinated prejudice . promises is a compelling piece that demonstrates just how well children can be trained to live out and carry on their parents' anguish .",
  'a film without surprise geared toward maximum comfort and familiarity .'],
 'label': [1, 0, 0, 1, 0]}

In [25]:
from tqdm import tqdm

def zero_shot_clf(review):
    prompt = f'Classify the following movie review as "positive" or "negative": "{review}\"'
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )
    return response

responses = []
for review in tqdm(examples['text']):
    response = zero_shot_clf(review)
    responses.append(response)

100%|██████████| 5/5 [00:02<00:00,  1.93it/s]


In [26]:
for example, response in zip(examples, responses):
    print(f"<<Review>> {example['text']}")
    print(f"<<Response>> {response['message']['content']}")
    print(f"<<Ground truth>> {id2label[example['label']]}")
    print("-"*80)

<<Review>> it's like an old warner bros . costumer jived with sex -- this could be the movie errol flynn always wanted to make , though bette davis , cast as joan , would have killed him .
<<Response>> This review is overwhelmingly **negative**.

The passage contains highly explicit and disturbing content – referencing a physical environment and a character’s discomfort – which is a clear indication of a very poor review. It’s not a review of the film itself, but rather a critique of the production environment and the portrayal of the film.
<<Ground truth>> positive
--------------------------------------------------------------------------------
<<Review>> [lin chung's] voice is rather unexceptional , even irritating ( at least to this western ear ) , making it awfully hard to buy the impetus for the complicated love triangle that develops between the three central characters .
<<Response>> Negative.

The review expresses a strong dislike for the film’s narrator’s voice and the pacing 

### Chain-of-thought

Usemos _chain-of-thought_ y un prompt que nos permita encontrar la respuesta final fácilmente.

In [27]:
import re

def cot_clf(review):
    prompt = (
        '''Classify the following movie review as "positive" or "negative".'''
        ''' First, reason step by step about the review.'''
        ''' Then, write your answer on the last line in the format "Final Classification: [positive/negative]".\n\n'''
        f'''Review: "{review}"'''
    )
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )
    return response

def extract_final_classification(response_text):
    lines = response_text.split('\n')
    for line in lines:
        if 'Final Classification:' in line:
            answer = line.split(':')[1].strip()
            answer = re.sub(r'[^\w\s]', '', answer.lower())
            return answer
    return None

In [28]:
responses = []
for review in tqdm(examples['text']):
    response = cot_clf(review)
    responses.append(response)

100%|██████████| 5/5 [00:04<00:00,  1.19it/s]


In [29]:
def print_results(examples, responses):
    for example, response in zip(examples, responses):
        print(f"Review: {example['text']}")
        text = response['message']['content']
        print(f"<<Response>> {text}")
        print(f"<<Final Classification>> {extract_final_classification(text)}")
        print(f"<<Ground truth>> {id2label[example['label']]}")
        print("-"*80)

print_results(examples, responses)

Review: it's like an old warner bros . costumer jived with sex -- this could be the movie errol flynn always wanted to make , though bette davis , cast as joan , would have killed him .
<<Response>> Okay, let's break down this review.

1. **Overall Tone:** The review is extremely provocative and uses strong, somewhat jarring language. It’s not a straightforward review of the film.

2. **Key Phrases:** The phrases "it's like an old Warner Bros. customer danced with sex" and "Bette Davis would have killed him" are the most significant. These phrases are highly unusual and suggest a significant disruption to the film’s atmosphere.

3. **Implied Criticism:** The reviewer is clearly expressing a strong negative opinion of the film. The repeated emphasis on a bizarre and unsettling experience suggests the reviewer found the film deeply unpleasant.

4. **Conclusion:** The reviewer is essentially saying the film is a mess, and that the experience was unpleasant.

**Final Classification: negati

## Prompt prefilling y stop sequences

Para obtener respuestas acotadas y parseables, podemos usar:

* **Prompt prefilling** consiste en pre-llenar el inicio de la respuesta del asistente para guiar la generación. En `chat()`, se agrega un mensaje con `"role": "assistant"` al final de la lista.
* Las **stop sequences** son strings que, al aparecer en la generación, hacen que el modelo deje de generar.

El prefilling es especialmente útil para forzar un formato de salida (JSON, XML, código) sin depender de que el modelo siga las instrucciones del prompt. La stop sequence complementa esto evitando que el modelo siga generando después del cierre esperado.

A continuación comparamos una respuesta sin estas técnicas con otra que combina ambas.

In [30]:
# Sin prefilling ni stop: el modelo responde libremente
review = "The film was so nice that I almost vomited."
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": f'Classify this movie review as "positive" or "negative".\n\nReview: "{review}"'}
    ]
)
print(response['message']['content'])

Negative.

The phrase "almost vomited" is a strong and negative reaction to the movie. It indicates a very unpleasant experience.


In [31]:
# Con prefilling + stop sequence:
# - Prefilling: forzamos que el modelo arranque razonando
# - Stop: cortamos en ``` para obtener solo las dos líneas
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": (
            f'Classify this movie review as "positive" or "negative".\n\n'
            f'Respond in exactly this format:\n'
            f'```\nReasoning: <brief reasoning>\n'
            f'Classification: <positive or negative>\n```\n\n'
            f'Review: "{review}"'
        )},
        {"role": "assistant", "content": "Reasoning: "}
    ],
    options={"stop": ["```"]}
)

print(response["message"]["content"])


The review uses an extreme and hyperbolic statement ("almost vomited") to express a strong negative reaction, indicating a very poor experience.
Classification: negative



## Clasificación few-shot

Puede ser útil pasar ejemplos de cómo se resuelve la tarea. En este caso elegimos los ejemplos aleatoriamente del set de entrenamiento y escribimos a mano el razonamiento de cada uno. También podemos elegir los ejemplos manualmente, o buscando ejemplos parecidos en el set de entrenamiento.

In [32]:
from datasets import concatenate_datasets

# Razonamientos escritos a mano para los ejemplos que selecciona get_few_shot_examples
reasonings = {
    "[a] shapeless blob of desperate entertainment .":
        "The reviewer calls the film a \"shapeless blob\" and describes the entertainment as \"desperate\", which are dismissive judgments with nothing positive to balance them.",
    "films about loss , grief and recovery are pretty valuable these days . seen in that light , moonlight mile should strike a nerve in many .":
        "The reviewer considers films about loss and grief valuable and expects Moonlight Mile to resonate with many viewers, which is an endorsement.",
    "it's super- violent , super-serious and super-stupid .":
        "The list of exaggerated descriptors ends with \"super-stupid\", which makes it clear that the review is mocking the film.",
    "ruzowitzky has taken this mothball-y stuff and made a rather sturdy , old-fashioned entertainment out of it .":
        "Although the source material is described as dated (\"mothball-y\"), the reviewer praises the director for turning it into sturdy, old-fashioned entertainment.",
}

def get_few_shot_examples(dataset, num_examples=4):
    examples_pos = dataset['train'].filter(lambda x: x['label'] == 1).shuffle(seed=36).select(range(num_examples // 2))
    examples_neg = dataset['train'].filter(lambda x: x['label'] == 0).shuffle(seed=36).select(range(num_examples // 2))
    examples = concatenate_datasets([examples_pos, examples_neg]).shuffle(seed=5)
    out = []
    for example in examples:
        label = id2label[example['label']]
        formatted_example = (
            f'''Review: "{example['text']}"'''
            f'''\nReasoning: {reasonings[example['text']]}'''
            f'''\nFinal Classification: {label}'''
        )
        out.append(formatted_example)
    return out

few_shot_examples = get_few_shot_examples(dataset)
for example in few_shot_examples:
    print(example)
    print("-"*80)

Filter:   0%|          | 0/8530 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8530 [00:00<?, ? examples/s]

Review: "[a] shapeless blob of desperate entertainment ."
Reasoning: The reviewer calls the film a "shapeless blob" and describes the entertainment as "desperate", which are dismissive judgments with nothing positive to balance them.
Final Classification: negative
--------------------------------------------------------------------------------
Review: "films about loss , grief and recovery are pretty valuable these days . seen in that light , moonlight mile should strike a nerve in many ."
Reasoning: The reviewer considers films about loss and grief valuable and expects Moonlight Mile to resonate with many viewers, which is an endorsement.
Final Classification: positive
--------------------------------------------------------------------------------
Review: "it's super- violent , super-serious and super-stupid ."
Reasoning: The list of exaggerated descriptors ends with "super-stupid", which makes it clear that the review is mocking the film.
Final Classification: negative
-------------

**PREGUNTA 4**: ¿De dónde se toman los ejemplos en few-shot? ¿Train, validation, test?

In [33]:
def cot_few_shot_prompt(review, few_shot_examples):
    prompt = (
        '''Classify the following movie review as "positive" or "negative".'''
        ''' First, reason step by step about the review.'''
        ''' Then, write your answer on the last line in the format "Final Classification: [positive/negative]".\n\n'''
    )
    prompt += 'Examples:'
    prompt += '\n------\n'
    prompt += '\n\n'.join(few_shot_examples)
    prompt += '\n------\n'
    prompt += f'\nReview: "{review}"'
    return prompt

print(cot_few_shot_prompt("EJEMPLO.", few_shot_examples))

Classify the following movie review as "positive" or "negative". First, reason step by step about the review. Then, write your answer on the last line in the format "Final Classification: [positive/negative]".

Examples:
------
Review: "[a] shapeless blob of desperate entertainment ."
Reasoning: The reviewer calls the film a "shapeless blob" and describes the entertainment as "desperate", which are dismissive judgments with nothing positive to balance them.
Final Classification: negative

Review: "films about loss , grief and recovery are pretty valuable these days . seen in that light , moonlight mile should strike a nerve in many ."
Reasoning: The reviewer considers films about loss and grief valuable and expects Moonlight Mile to resonate with many viewers, which is an endorsement.
Final Classification: positive

Review: "it's super- violent , super-serious and super-stupid ."
Reasoning: The list of exaggerated descriptors ends with "super-stupid", which makes it clear that the revi

In [34]:
def cot_few_shot_clf(review, few_shot_examples):
    prompt = cot_few_shot_prompt(review, few_shot_examples)
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )
    return response

In [35]:
responses = []
for review in tqdm(examples['text']):
    response = cot_few_shot_clf(review, few_shot_examples)
    responses.append(response)

100%|██████████| 5/5 [00:02<00:00,  2.12it/s]


In [36]:
print_results(examples, responses)

Review: it's like an old warner bros . costumer jived with sex -- this could be the movie errol flynn always wanted to make , though bette davis , cast as joan , would have killed him .
<<Response>> Reasoning: The review expresses admiration for Errol Flynn’s previous work and suggests the film could be a significant and successful project, highlighting Flynn’s vision. It also acknowledges the strong performance of Bettie Davis as Joan.
Final Classification: positive
<<Final Classification>> positive
<<Ground truth>> positive
--------------------------------------------------------------------------------
Review: [lin chung's] voice is rather unexceptional , even irritating ( at least to this western ear ) , making it awfully hard to buy the impetus for the complicated love triangle that develops between the three central characters .
<<Response>> Reasoning: The reviewer highlights the perceived lack of vocal quality in Lin Chung’s performance ("rather unexceptional, even irritating") 

**PREGUNTA 5** ¿Cómo probarías en un caso real qué tipo de prompt o qué modelo es mejor?

## Extracción de datos estructurados de texto libre

Veamos cómo extraer los ingredientes de una receta.

In [37]:
ds_recetas = load_dataset("somosnlp/RecetasDeLaAbuela", "version_1")

README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

main.csv: reconstructing file:   0%|          |  0.00B / 40.3MB            

main.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20236 [00:00<?, ? examples/s]

In [38]:
print(ds_recetas)

DatasetDict({
    train: Dataset({
        features: ['Id', 'Nombre', 'URL', 'Ingredientes', 'Pasos', 'Pais', 'Duracion', 'Categoria', 'Contexto', 'Valoracion y Votos', 'Comensales', 'Tiempo', 'Dificultad', 'Valor nutricional'],
        num_rows: 20236
    })
})


In [39]:
ds_small = ds_recetas["train"].filter(lambda x: x["Pais"] == "ARG").shuffle(seed=33).select(range(5))

Filter:   0%|          | 0/20236 [00:00<?, ? examples/s]

In [40]:
import ast
import re

def preprocess(example):
    if example["Pasos"].startswith("["):
        pasos_list = ast.literal_eval(example["Pasos"].encode("unicode_escape").decode("ascii"))
        example["Pasos"] = " ".join(pasos_list)
    example["Pasos"] = re.sub(r'\s+', ' ', example["Pasos"])
    return example

ds_small = ds_small.map(preprocess)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [41]:
for example in ds_small:
    print(example["Ingredientes"])
    print(example["Pasos"])
    print("-"*80)

['1 paquete de galletitas de chocolate (230 gramos)', '400 gramos de dulce de leche tradicional', '400 gramos de queso untable', '100 centímetros cúbicos de leche', '1 cucharada postre de café instantáneo', '1 paquete de galletitas rellenas (90 gramos)', '4 cucharadas de postre de dulce de leche', '100 gramos de frutillas']
1 El primer paso para hacer esta torta de chocotorta decorada consiste en mezclar el dulce de leche con el queso untable. Aquí combinamos la misma cantidad de uno y otro, pero puedes hacerlo con más queso o más dulce. También puedes elegir un queso descremado o entero. 2 Mezcla la leche con el café instantáneo. En él, humedecerás cada galletita antes de armar con ellas la base de la chocotorta. Puedes mezclar café con cacao dulce o reemplazarlo por completo si haces esta torta para chicos. Truco: el café resaltará el sabor del chocolate y bajará un poco el punto dulce de la torta. 3 Para armar la torta, elige un molde redondo, en este caso de 16 centímetros de diáme

In [42]:
def prompt_extract_json(receta):
    prompt = (
        '''Extraer los ingredientes de la siguiente receta en formato JSON.'''
        f'''\nReceta: {receta}'''
    )
    return prompt

print(prompt_extract_json(ds_small[0]["Pasos"]))

Extraer los ingredientes de la siguiente receta en formato JSON.
Receta: 1 El primer paso para hacer esta torta de chocotorta decorada consiste en mezclar el dulce de leche con el queso untable. Aquí combinamos la misma cantidad de uno y otro, pero puedes hacerlo con más queso o más dulce. También puedes elegir un queso descremado o entero. 2 Mezcla la leche con el café instantáneo. En él, humedecerás cada galletita antes de armar con ellas la base de la chocotorta. Puedes mezclar café con cacao dulce o reemplazarlo por completo si haces esta torta para chicos. Truco: el café resaltará el sabor del chocolate y bajará un poco el punto dulce de la torta. 3 Para armar la torta, elige un molde redondo, en este caso de 16 centímetros de diámetro. Cúbrelo con papel film para que puedas desmoldar luego. Haz una capa base con galletitas húmedas y, luego, agrega el relleno de dulce con queso para hacer otra capa de galletitas hasta utilizar todo. La última capa debería ser de galletitas. 4 Llev

In [43]:
import json

# Los modelos chicos a veces se traban repitiendo el mismo token con format="json"
# (por ejemplo, espacios o saltos de línea). Ollama corta la generación y lanza un
# ResponseError ("token repeat limit reached"). Capturamos el error y devolvemos una
# respuesta vacía con la misma forma, para que el resto del procesamiento siga funcionando.
def safe_chat(**kwargs):
    try:
        return ollama.chat(**kwargs)
    except ollama.ResponseError as e:
        print(f"Error de Ollama: {e.error}")
        return {"message": {"content": ""}}

def extract_json(receta):
    prompt = prompt_extract_json(receta)
    response = safe_chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        format="json",
        options={"num_predict": 512}  # tope de tokens por si el modelo se traba
    )
    return response

In [44]:
responses = []
for receta in tqdm(ds_small['Pasos']):
    response = extract_json(receta)
    responses.append(response)

100%|██████████| 5/5 [00:05<00:00,  1.13s/it]


In [45]:
for example, response in zip(ds_small, responses):
    print(f"Receta: {example['Pasos']}")
    print(f"Response: {response['message']['content']}")
    print("-"*80)

Receta: 1 El primer paso para hacer esta torta de chocotorta decorada consiste en mezclar el dulce de leche con el queso untable. Aquí combinamos la misma cantidad de uno y otro, pero puedes hacerlo con más queso o más dulce. También puedes elegir un queso descremado o entero. 2 Mezcla la leche con el café instantáneo. En él, humedecerás cada galletita antes de armar con ellas la base de la chocotorta. Puedes mezclar café con cacao dulce o reemplazarlo por completo si haces esta torta para chicos. Truco: el café resaltará el sabor del chocolate y bajará un poco el punto dulce de la torta. 3 Para armar la torta, elige un molde redondo, en este caso de 16 centímetros de diámetro. Cúbrelo con papel film para que puedas desmoldar luego. Haz una capa base con galletitas húmedas y, luego, agrega el relleno de dulce con queso para hacer otra capa de galletitas hasta utilizar todo. La última capa debería ser de galletitas. 4 Lleva la chocotorta argentina a la heladera por al menos unos 60 minu

In [46]:
def prompt_extract_json2(receta):
    prompt = (
        '''Extraer los ingredientes de la siguiente receta en formato JSON.'''
        ''' El JSON debe contener únicamente la clave "ingredientes".'''
        f'''\nReceta: {receta}'''
    )
    return prompt

def extract_json2(receta):
    prompt = prompt_extract_json2(receta)
    response = safe_chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        format="json",
        options={"num_predict": 512}
        # constrained decoding: solo permite tokens que mantengan el texto como un JSON válido.
        # Si la generación se corta antes de cerrar el JSON, el resultado puede quedar incompleto.
        # Para fijar también las claves y los tipos se puede pasar un JSON schema:
        # format=Modelo.model_json_schema() (con una clase de pydantic)
        # La alternativa menos segura es usar prefilling y stop sequences
    )
    return response

responses = []
for receta in tqdm(ds_small['Pasos']):
    response = extract_json2(receta)
    responses.append(response)

100%|██████████| 5/5 [00:03<00:00,  1.46it/s]


In [47]:
results = []
for response in responses:
    try:
        data = json.loads(response['message']['content'])
        results.append(data)
    except Exception as e:
        print(f"Error: {e}")

In [48]:
results[0]

{'ingredientes': 'Dulce de leche, queso untable, leche, café instantáneo, galletas, queso descremado o entero, cacao dulce o café, galletas (para el molde)'}

In [49]:
import pandas as pd

df = pd.DataFrame(results)
df.head()

,ingredientes
0,"Dulce de leche, queso untable, leche, café ins..."
1,"[maíz, leche, azúcar, vainilla, canela, pan du..."
2,"[1. Cebolla, 2. Aceite, 3. Manteca, 4. Vino bl..."
3,[Corta la bondiola en churrasquitos de un dedo...
4,"[carne (pollo, cerdo, vaca, lomo, bola de lomo..."


In [50]:
df['ingredientes'].explode().value_counts()

,count
ingredientes,
"Dulce de leche, queso untable, leche, café instantáneo, galletas, queso descremado o entero, cacao dulce o café, galletas (para el molde)",1
maíz,1
leche,1
azúcar,1
vainilla,1
canela,1
pan dulce argentino,1
1. Cebolla,1
2. Aceite,1


In [51]:
def prompt_extract_json3(receta):
    prompt = (
        '''Extraer los ingredientes de la siguiente receta en formato JSON.'''
        ''' El JSON debe contener las claves "veganos" y "no_veganos".'''
        ''' Ambas claves deben contener una lista de ingredientes.'''
        ''' Si no hay ingredientes para alguna clave, se debe incluir una lista vacía.'''
        f'''\nReceta: {receta}'''
    )
    return prompt

def extract_json3(receta):
    prompt = prompt_extract_json3(receta)
    response = safe_chat(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        format="json",
        options={"num_predict": 512}
    )
    return response

responses = []
for receta in tqdm(ds_small['Pasos']):
    response = extract_json3(receta)
    responses.append(response)

100%|██████████| 5/5 [00:03<00:00,  1.53it/s]


In [52]:
results = []
for response in responses:
    try:
        data = json.loads(response['message']['content'])
        results.append(data)
    except Exception as e:
        print(f"Error: {e}")

df = pd.DataFrame(results)
df.head()

,veganos,no_veganos
0,"[Dulce de leche, Queso untable, Queso descrema...","[Leche, Café instantáneo, Galletitas, Manga o ..."
1,[leche],[maíz]
2,"[Sí, No]","[Sí, No]"
3,"[Bondiola, Aceite]",[Vino blanco]
4,"[Huevo batido con leche, Pimienta, provenzal o...","[Cerveza, Miel, Sandia, Aceite, Agua, Bicarbon..."


In [53]:
df['veganos'].explode().value_counts()
# Recomendación: en general es mejor descomponer las tareas en pasos
# Acá: primero identificar componentes, luego clasificar en vegano/no-vegano

,count
veganos,
Dulce de leche,1
Queso untable,1
Queso descremado,1
Chocolate instantáneo,1
leche,1
Sí,1
No,1
Bondiola,1
Aceite,1


## Conversaciones multi-turno

Con Ollama manejamos el historial de mensajes manualmente.

In [54]:
history = []

def chat_send(message, history):
    history.append({"role": "user", "content": message})
    response = ollama.chat(
        model=MODEL_NAME,
        messages=history
    )
    assistant_msg = response['message']['content']
    history.append({"role": "assistant", "content": assistant_msg})
    return assistant_msg

In [55]:
print(f"{len(history)=}")
response_text = chat_send("¿Por qué el agua es azul?", history)
to_markdown(response_text)

len(history)=0


> La respuesta corta es que el agua **no es azul** en sí misma, sino que **absorbe y refleja la luz azul**. Es una propiedad que se ve afectada por la forma en que la luz interactúa con el agua. Aquí te explico más en detalle:
> 
> 
> 
> **1. La Absorción de la Luz:**
> 
> 
> 
> * **El espectro de luz:** La luz blanca se compone de todos los colores del arco iris. Cada color tiene una longitud de onda diferente.
> 
> * **El agua absorbe:** El agua absorbe principalmente la luz azul y violeta, mientras que refleja y transmite más la luz roja, naranja y amarilla. Es como si el agua tuviera "mordido" en esa parte del espectro.
> 
> 
> 
> **2. ¿Por qué la vemos azul?**
> 
> 
> 
> * **Reflexión:** Cuando la luz viaja a través del agua, los diminutos cristales de hidrocarburo (la estructura cristalina del agua) dispersan la luz.  Esta dispersión hace que la luz azul se refleje en la superficie del agua.
> 
> * **El color que percibimos:**  Nuestros ojos son más sensibles a la luz azul que a la luz roja.  Por lo tanto, cuando vemos agua, estamos viendo principalmente la luz azul que ha sido reflejada.
> 
> 
> 
> **3. Diferentes tonalidades de azul:**
> 
> 
> 
> * **Agua más clara (como el agua del mar):** Tiene una absorción de luz azul más alta, lo que hace que se vea más blanca.
> 
> * **Agua más oscura (como el agua de un lago):** Tiene una absorción de luz azul más baja, lo que la hace parecer más oscura.
> 
> 
> 
> **En resumen:** El agua parece azul debido a la forma en que la luz interactúa con sus cristales, causando que se refleje y refleje la luz azul.
> 
> 
> 
> **Ejemplo:**
> 
> 
> 
> Piensa en un lago en un día soleado.  La luz solar incide en el agua, y la luz azul es absorvida, permitiendo que la luz roja y naranja lleguen a nuestros ojos.  Esta combinación de colores hace que veamos el agua como azul.
> 
> 
> 
> **Observaciones:**
> 
> 
> 
> * **El color del agua puede variar:**  El color exacto del agua puede variar según factores como la temperatura, la turbidez del agua, y la presencia de partículas en suspensión.
> 
> * **No es "azul por naturaleza":** El agua no es inherentemente azul en su estado natural. Es la forma en que la luz interactúa con ella.
> 
> 
> 
> Espero que esta explicación te ayude a comprender por qué el agua parece azul!
> 


In [56]:
print(f"{len(history)=}")
response_text = chat_send("¿Cómo se lo explicarías a un niño de 8 años?", history)
to_markdown(response_text)

len(history)=2


> ¡Claro! Aquí te lo explico con palabras que un niño de 8 años entienda:
> 
> 
> 
> "Imagina que el agua es como un lienzo donde los colores se mezclan. Cuando la luz del sol entra en el agua, ¡es como si el agua estuviera absorbiendo algunos de los colores, especialmente el azul!  
> 
> 
> 
> Es como si tuviera un pequeño 'mordisco' en esas partes del color que queremos ver, así que el azul es lo que vemos más fuerte. 
> 
> 
> 
> Así que, cuando vemos el agua, es porque la luz azul se está volviendo a nosotros. ¡Así que el agua se ve azul!"
> 
> 
> 
> **Para hacerlo aún más divertido:**
> 
> 
> 
> *   Podrías usar una comparación con un truco de luces: "El agua es un poco como cuando haces un truco de luces, pero en lugar de hacer un truco, la luz se "pasa" por el agua".
> 
> *   Puedes preguntarle: "¿Qué te parece si observamos el agua en un día soleado?"
> 
> 
> 
> Espero que esto sea lo que necesitas. ¡Dime si quieres otra explicación!

In [57]:
print(f"{len(history)=}")
for message in history:
    display(to_markdown(f"**{message['role']}**: {message['content']}"))

len(history)=4


> **user**: ¿Por qué el agua es azul?

> **assistant**: La respuesta corta es que el agua **no es azul** en sí misma, sino que **absorbe y refleja la luz azul**. Es una propiedad que se ve afectada por la forma en que la luz interactúa con el agua. Aquí te explico más en detalle:
> 
> 
> 
> **1. La Absorción de la Luz:**
> 
> 
> 
> * **El espectro de luz:** La luz blanca se compone de todos los colores del arco iris. Cada color tiene una longitud de onda diferente.
> 
> * **El agua absorbe:** El agua absorbe principalmente la luz azul y violeta, mientras que refleja y transmite más la luz roja, naranja y amarilla. Es como si el agua tuviera "mordido" en esa parte del espectro.
> 
> 
> 
> **2. ¿Por qué la vemos azul?**
> 
> 
> 
> * **Reflexión:** Cuando la luz viaja a través del agua, los diminutos cristales de hidrocarburo (la estructura cristalina del agua) dispersan la luz.  Esta dispersión hace que la luz azul se refleje en la superficie del agua.
> 
> * **El color que percibimos:**  Nuestros ojos son más sensibles a la luz azul que a la luz roja.  Por lo tanto, cuando vemos agua, estamos viendo principalmente la luz azul que ha sido reflejada.
> 
> 
> 
> **3. Diferentes tonalidades de azul:**
> 
> 
> 
> * **Agua más clara (como el agua del mar):** Tiene una absorción de luz azul más alta, lo que hace que se vea más blanca.
> 
> * **Agua más oscura (como el agua de un lago):** Tiene una absorción de luz azul más baja, lo que la hace parecer más oscura.
> 
> 
> 
> **En resumen:** El agua parece azul debido a la forma en que la luz interactúa con sus cristales, causando que se refleje y refleje la luz azul.
> 
> 
> 
> **Ejemplo:**
> 
> 
> 
> Piensa en un lago en un día soleado.  La luz solar incide en el agua, y la luz azul es absorvida, permitiendo que la luz roja y naranja lleguen a nuestros ojos.  Esta combinación de colores hace que veamos el agua como azul.
> 
> 
> 
> **Observaciones:**
> 
> 
> 
> * **El color del agua puede variar:**  El color exacto del agua puede variar según factores como la temperatura, la turbidez del agua, y la presencia de partículas en suspensión.
> 
> * **No es "azul por naturaleza":** El agua no es inherentemente azul en su estado natural. Es la forma en que la luz interactúa con ella.
> 
> 
> 
> Espero que esta explicación te ayude a comprender por qué el agua parece azul!
> 


> **user**: ¿Cómo se lo explicarías a un niño de 8 años?

> **assistant**: ¡Claro! Aquí te lo explico con palabras que un niño de 8 años entienda:
> 
> 
> 
> "Imagina que el agua es como un lienzo donde los colores se mezclan. Cuando la luz del sol entra en el agua, ¡es como si el agua estuviera absorbiendo algunos de los colores, especialmente el azul!  
> 
> 
> 
> Es como si tuviera un pequeño 'mordisco' en esas partes del color que queremos ver, así que el azul es lo que vemos más fuerte. 
> 
> 
> 
> Así que, cuando vemos el agua, es porque la luz azul se está volviendo a nosotros. ¡Así que el agua se ve azul!"
> 
> 
> 
> **Para hacerlo aún más divertido:**
> 
> 
> 
> *   Podrías usar una comparación con un truco de luces: "El agua es un poco como cuando haces un truco de luces, pero en lugar de hacer un truco, la luz se "pasa" por el agua".
> 
> *   Puedes preguntarle: "¿Qué te parece si observamos el agua en un día soleado?"
> 
> 
> 
> Espero que esto sea lo que necesitas. ¡Dime si quieres otra explicación!

**PREGUNTA 6**: ¿Qué pasa si en chat_send no agregamos la respuesta del asistente al historial?

## `generate()` vs. `chat()`

Hay dos interfaces para hacer inferencia con los modelos:

* `ollama.generate()` recibe un string como prompt. Por defecto, Ollama le aplica el template de chat del modelo (como si fuera un único mensaje del usuario); con `raw=True` el string se pasa tal cual y el modelo genera una continuación (completion).
* `ollama.chat()` es la interfaz conversacional: recibe una lista de mensajes con roles (`system`, `user`, `assistant`) y el modelo responde respetando el template de chat con el que fue entrenado.
* `chat()` es más cómodo para conversaciones multi-turno
* `generate()` con raw=True es útil cuando se quiere control total sobre el prompt exacto que recibe el modelo. Cada familia de modelos usa un formato distinto.
* Para hacer prompt prefilling en `generate()`, se usa `raw=True` y se incluye el inicio de la respuesta al final del prompt, después de la marca de turno del modelo.

In [58]:
# El template del modelo:
!ollama show --modelfile {MODEL_NAME} | grep -A 20 "TEMPLATE"

TEMPLATE """{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if or (eq .Role "user") (eq .Role "system") }}<start_of_turn>user
{{ .Content }}<end_of_turn>
{{ if $last }}<start_of_turn>model
{{ end }}
{{- else if eq .Role "assistant" }}<start_of_turn>model
{{ .Content }}{{ if not $last }}<end_of_turn>
{{ end }}
{{- end }}
{{- end }}"""
PARAMETER top_k 64
PARAMETER top_p 0.95
PARAMETER stop <end_of_turn>
PARAMETER temperature 1
LICENSE """Gemma Terms of Use 

Last modified: February 21, 2024

By using, reproducing, modifying, distributing, performing or displaying any portion or element of Gemma, Model Derivatives including via any Hosted Service, (each as defined below) (collectively, the "Gemma Services") or otherwise accepting the terms of this Agreement, you agree to be bound by this Agreement.



In [59]:
# Define cómo Ollama convierte la lista de mensajes en el prompt que recibe el modelo.
# * Si el rol es user o system → lo envuelve en <start_of_turn>user ... <end_of_turn>
# * Si el rol es assistant → lo envuelve en <start_of_turn>model ... <end_of_turn>
# * Si el último msje es del usuario, agrega <start_of_turn>model al final
#   (para que el modelo empiece a generar su turno)
# * Si el último es del assistant, no cierra con <end_of_turn>
#   (para que el modelo siga desde ahí, esto es lo que habilita el prefilling)
# * El PARAMETER stop le dice a Ollama que corte la generación cuando el modelo
#   genera <end_of_turn>, para que no invente turnos adicionales.

In [60]:
# chat(): recibe una lista de mensajes con roles
response_chat = ollama.chat(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "¿Qué es la física cuántica?"}]
)
print(response_chat['message']['content'])



La física cuántica es una rama fundamental de la física que estudia el comportamiento de la materia y la energía a un nivel atómico y subatómico. Es una de las teorías más revolucionarias y, a menudo, contraintuitivas de la ciencia moderna. A diferencia de la física clásica (que describe el mundo con un comportamiento determinista y predecible), la física cuántica introduce conceptos radicalmente diferentes, como la cuantización, la superposición y el principio de incertidumbre.

Aquí te dejo una explicación más detallada de los aspectos clave:

**1. La Revolución de la Cuantización:**

* **La idea central:** La física cuántica postula que la energía, el tiempo y otros atributos físicos no pueden tomar cualquier valor continuo, sino que se encuentran en "paquetes" discretos llamados "cuantos".  Imagina que la energía no es como una rampa que se puede subir gradualmente, sino como una escalera con escalones que solo puedes subir a un número específico de escalones.
* **Ejemplo:** La luz

In [61]:
# generate() sin raw=True: Ollama aplica el template de chat al prompt,
# por eso la salida es una respuesta y no una continuación del texto
response_gen = ollama.generate(
    model=MODEL_NAME,
    prompt="¿Qué es la física cuántica?"
)
print(response_gen['response'])

La física cuántica es una rama de la física que se ocupa del comportamiento de la materia y la energía a nivel atómico y subatómico.  Es una teoría fundamental que describe el mundo a escala atómica y subatómica, y **desafía nuestra intuición sobre el mundo que percibimos a través de los sentidos**. En lugar de ver las cosas como entidades definidas y separadas, la física cuántica nos dice que la realidad a nivel fundamental es probabilística y fundamentalmente cuantizada.

Aquí te desglosamos los puntos clave:

**1. La Naturaleza Cuantizada:**

* **Cuantización:**  La idea más fundamental es que la energía, el momento, la angularidad y otras propiedades físicas no pueden tomar cualquier valor, sino que solo pueden tomar valores discretos y específicos, como los pasos en una escalera.  Piensa en ello como si la energía solo existiera en 'picos' en lugar de un rango continuo.  Esta es la noción central de la cuantización.
* **Partículas vs. Onda:**  Un concepto crucial es que las partíc

In [62]:
# Para Gemma, el template es:
# <start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n
prompt_con_marcas = """<start_of_turn>user
¿Qué es la física cuántica?<end_of_turn>
<start_of_turn>model
"""

response = ollama.generate(
    model=MODEL_NAME,
    prompt=prompt_con_marcas,
    raw=True  # raw=True evita que Ollama aplique el template de nuevo
)
print(response['response'])

La física cuántica es una rama fundamental de la física que se ocupa del estudio del comportamiento de la materia y la energía a nivel atómico y subatómico. Es una teoría que describe el universo a la escala más pequeña, donde las reglas de la física clásica (las que entendemos en nuestra vida cotidiana) dejan de funcionar.  Es una de las teorías más revolucionarias y complejas de la ciencia moderna, y tiene implicaciones profundas en muchas áreas de la vida.

Aquí te desglosamos los puntos clave de lo que es la física cuántica:

**1. La Naturaleza Cuántica:**

* **Cuantización:** La idea central es que la energía, el espacio y otras propiedades físicas no son continuos, sino que solo pueden existir en cantidades discretas, llamadas "cuantos".  Piensa en ello como un escalón en lugar de una rampa.  Es como si un átomo solo pudiera tener ciertos niveles de energía, y no pudiera existir en cualquier nivel.
* **Dualidad Onda-Partícula:** Una de las ideas más sorprendentes es que las partí

----------------------------

## Extras

* **Streaming**: con `stream=True` podemos recibir la respuesta token a token en lugar de esperar a que se genere completa. Esto es útil para interfaces interactivas.

* **Cuantización**: los modelos en la library de Ollama suelen estar disponibles en distintos niveles de cuantización: `q4_K_M`, `q8_0`, `fp16`, etc. Esto reduce la precisión numérica de los pesos del modelo (por ejemplo, de 16 bits a 4 bits), lo que disminuye el uso de memoria y acelera la inferencia a costa de una leve pérdida de calidad. En la práctica, las variantes `q4` suelen ser un buen compromiso: ocupan ~3x menos memoria que `fp16` con una degradación muy leve en la mayoría de las tareas.

Por ejemplo:

```bash
ollama pull gemma3:1b              # cuantización default (q4_K_M)
ollama pull gemma3:1b-it-fp16      # precisión completa
ollama pull gemma3:1b-it-q8_0      # cuantización a 8 bits
```

* Ollama también tiene modelos **multimodales** que pueden recibir imágenes además de texto. Se pasan las imágenes en el parámetro `images` de `ollama.chat()`.

* También se pueden generar embeddings con `ollama.embed()`. En principio podemos obtener los embeddings de cualquier modelo, pero esto tiene sentido cuando queremos vectores optimizados para una tarea basada en similitud semántica; e.g. clustering.

* **[vLLM](https://docs.vllm.ai/)** es un motor de inferencia optimizado para alto throughput. Ollama está pensado para uso local e interactivo (puede atender algunos requests en paralelo, configurable con `OLLAMA_NUM_PARALLEL`), mientras que vLLM está pensado para servir modelos en producción o procesar grandes volúmenes de datos. Su ventaja principal es el **batching eficiente**: con PagedAttention la KV cache se asigna en bloques según los tokens que realmente usa cada secuencia, y con continuous batching puede procesar cientos de prompts en paralelo aprovechando la GPU al máximo. Para tareas como clasificar 1000 reviews, vLLM es significativamente más rápido que hacer 1000 llamadas secuenciales a Ollama. El parámetro `max_model_len` fija la longitud máxima (prompt + respuesta) de cada secuencia; vLLM verifica al iniciar que una secuencia de esa longitud entre en la KV cache, así que conviene ajustarlo a lo que necesita la tarea en lugar de usar el default del modelo. En nuestro caso, las reviews tienen ~25 palabras en promedio, así que `max_model_len=1024` es más que suficiente.